# Unified Dataset EDA

Exploratory data analysis of the unified dataset containing math problems with text and labels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# Load the dataset
df = pd.read_json('../data/raw/unified_dataset.jsonl', lines=True)

print(f"Dataset shape: {df.shape}")
df.head()

## Schema and Missingness

Overview of column types and missing value patterns.

In [ ]:
# Schema overview
print("Column dtypes:")
print(df.dtypes)
print("\n" + "="*50 + "\n")

# Missingness summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'non_null': len(df) - missing, 'missing': missing, 'missing_pct': missing_pct})
print("Missingness summary:")
print(missing_df)

## Label Distribution

Analysis of the numeric label column (all 30 rows have labels).

In [ ]:
# Label distribution
if 'label' in df.columns and df['label'].notna().any():
    print(f"Label stats:\n{df['label'].describe()}\n")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    df['label'].hist(bins=15, ax=axes[0], edgecolor='black')
    axes[0].set_title('Label Distribution')
    axes[0].set_xlabel('Label Value')
    axes[0].set_ylabel('Frequency')
    
    # Box plot
    df['label'].plot.box(ax=axes[1])
    axes[1].set_title('Label Box Plot')
    axes[1].set_ylabel('Label Value')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nUnique labels: {df['label'].nunique()}")
    print(f"Top 10 most common labels:\n{df['label'].value_counts().head(10)}")

## Text Analysis

Character/word length distribution and top words from the text column.

In [ ]:
# Text length analysis
if 'text' in df.columns and df['text'].notna().any():
    # Calculate lengths
    df['char_len'] = df['text'].str.len()
    df['word_len'] = df['text'].str.split().str.len()
    
    print(f"Text length stats (characters):\n{df['char_len'].describe()}\n")
    print(f"Text length stats (words):\n{df['word_len'].describe()}\n")
    
    # Plot lengths
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['char_len'].hist(bins=15, ax=axes[0], edgecolor='black')
    axes[0].set_title('Character Length Distribution')
    axes[0].set_xlabel('Characters')
    
    df['word_len'].hist(bins=15, ax=axes[1], edgecolor='black')
    axes[1].set_title('Word Length Distribution')
    axes[1].set_xlabel('Words')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Top words analysis
if 'text' in df.columns and df['text'].notna().any():
    # Combine all text and extract words (lowercase, remove punctuation)
    all_text = ' '.join(df['text'].fillna('').astype(str)).lower()
    words = re.findall(r'\b[a-z]+\b', all_text)
    
    # Filter out common stop words
    stop_words = {'the', 'and', 'to', 'of', 'a', 'in', 'is', 'that', 'for', 'be', 'as', 'at', 'from', 'with', 'by', 'an', 'are', 'was', 'were', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'must', 'shall', 'can', 'need', 'dare', 'ought', 'used', 'to', 'it', 'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she', 'we', 'they', 'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'her', 'its', 'our', 'their', 'mine', 'yours', 'hers', 'ours', 'theirs'}
    filtered_words = [w for w in words if w not in stop_words and len(w) > 2]
    
    word_counts = Counter(filtered_words)
    top_words = word_counts.most_common(15)
    
    print("Top 15 words (excluding stop words):")
    for word, count in top_words:
        print(f"  {word}: {count}")
    
    # Plot top words
    if top_words:
        words_list, counts_list = zip(*top_words)
        plt.figure(figsize=(10, 5))
        plt.bar(words_list, counts_list, edgecolor='black')
        plt.title('Top 15 Words (Filtered)')
        plt.xlabel('Word')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()